# v18 物理セレクタ: 『最も物理的に正しい候補を真とする』

GRも学習も使わず、**物理的妥当性だけ**で PF 候補を選ぶ(教師なし)。
構造深度 `geo_z = TVT + Z` は地質構造なので **滑らか・dipは既知ゾーンと整合・typewell範囲内・非振動** が物理的に正しい。

物理項(各候補ごと, 小さいほど物理的):
- `rough` : geo_z の2階差RMS(曲率=dip不連続)
- `jerk`  : geo_z の3階差RMS(さらに高次の凸凹)
- `wiggle`: 傾きの符号反転率(振動=非物理ノイズ)
- `dipdev`: 平均dipと既知dipの差(構造dipは局所一定)
- `dipvar`: dipのばらつき
- `clip`  : typewell端への張り付き率(範囲外=実在しない)
- `anchor`: PS点での geo_z 連続性

各項を坑井内で標準化→重み付き和を物理スコアとし、**最小(=最も物理的)の候補を選択**。
単項ごと/組合せ/top-k を比較。基準: oracle≈4.62 / lik-ens≈9.35 / v9=6.946。

> ★注意: 『滑らかさ』単独だと直線(=動かない)に退化しがち。dip整合・非振動と組合せる。

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
PART_DIR = r'C:\Users\kosaka256\Documents\rogii_claude\v18\artifacts_v18\particles'
DATA_DIR = r'C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction\train'

WELLS = []
for f in sorted(glob.glob(os.path.join(PART_DIR, '*.npz'))):
    d = np.load(f); w = {k: d[k] for k in d.files}; w['wid'] = os.path.basename(f)[:-4]; WELLS.append(w)
print('wells:', len(WELLS))

def load_known(wid):
    try:
        hw = pd.read_csv(f'{DATA_DIR}/{wid}__horizontal_well.csv', usecols=['MD', 'Z', 'TVT_input'])
        kn = hw[hw['TVT_input'].notna()]
        if len(kn) < 3: return 0.0, np.nan
        g = (kn['TVT_input'] + kn['Z']).to_numpy(float); md = kn['MD'].to_numpy(float)
        tl = slice(max(0, len(g) - 30), len(g)); dm = np.diff(md[tl]); dg = np.diff(g[tl]); ok = dm > 0
        return (float(np.median(dg[ok] / dm[ok])) if ok.sum() >= 3 else 0.0), float(g[-1])
    except Exception:
        return 0.0, np.nan
KNOWN = {w['wid']: load_known(w['wid']) for w in WELLS}

def pooled_rmse(errs):
    e = np.concatenate(errs); e = e[np.isfinite(e)]; return float(np.sqrt(np.mean(e ** 2)))
def lik_ensemble(w, scale=20.0):
    lk = w['loglik'] - np.nanmax(w['loglik']); ww = np.exp(lk / scale); ww /= max(ww.sum(), 1e-300)
    return (ww[:, None] * w['cand']).sum(0)

# 基準
eo, ee = [], []
for w in WELLS:
    t = w['true']; m = np.isfinite(t)
    if not m.any(): continue
    rc = np.sqrt(np.mean((w['cand'][:, m] - t[None, m]) ** 2, 1))
    eo.append(w['cand'][int(np.argmin(rc)), m] - t[m]); ee.append(lik_ensemble(w)[m] - t[m])
ORACLE, LIKENS = pooled_rmse(eo), pooled_rmse(ee)
print(f'oracle={ORACLE:.3f}  lik-ensemble={LIKENS:.3f}')

In [ ]:
# ===== 物理項(候補ごと, 小さいほど物理的) =====
TERMS = ['rough', 'jerk', 'wiggle', 'dipdev', 'dipvar', 'clip', 'anchor']

def phys_terms(w):
    cand = w['cand'].astype(float); z = w['z'].astype(float); md = w['md'].astype(float)
    twt = w['tw_tvt'].astype(float); n = cand.shape[0]
    geo = cand + z[None, :]; d1 = np.diff(cand, axis=1)
    dip = np.gradient(geo, md, axis=1)
    kd, ag = KNOWN[w['wid']]; twmin, twmax = float(twt.min()), float(twt.max())
    return {
        'rough':  np.sqrt(np.mean(np.diff(geo, 2, axis=1) ** 2, 1)),
        'jerk':   np.sqrt(np.mean(np.diff(geo, 3, axis=1) ** 2, 1)),
        'wiggle': (np.abs(np.diff(np.sign(d1), axis=1)) > 0).mean(1),
        'dipdev': np.abs(dip.mean(1) - kd),
        'dipvar': dip.std(1),
        'clip':   ((cand <= twmin + 3) | (cand >= twmax - 3)).mean(1),
        'anchor': np.abs(geo[:, 0] - (ag if np.isfinite(ag) else geo[:, 0])),
    }

def zwell(v):
    v = np.asarray(v, float); return (v - v.mean()) / (v.std() + 1e-9)   # 坑井内で標準化

PHYS = [phys_terms(w) for w in WELLS]   # 事前計算
# lik-ensemble からの距離(RMSE, 全評価行)。restrict_R で近傍候補に限定するため。
ENS = [lik_ensemble(w) for w in WELLS]
ENS_DIST = [np.sqrt(np.mean((w['cand'] - ENS[wi][None, :]) ** 2, 1)) for wi, w in enumerate(WELLS)]

def allowed_idx(wi, restrict_R):
    if restrict_R is None:
        return np.arange(WELLS[wi]['cand'].shape[0])
    a = np.where(ENS_DIST[wi] <= restrict_R)[0]
    return a if a.size else np.array([int(np.argmin(ENS_DIST[wi]))])   # 0本なら最近傍1本

def select_overall(weights, k=1, restrict_R=None):
    """物理スコア=Σ w*z(term) 最小の候補を坑井ごとに選ぶ(top-k平均)。
    restrict_R: lik-ensemble から RMSE この値以内の候補だけを対象にする(Noneで全候補)。"""
    errs = []
    for wi, w in enumerate(WELLS):
        t = w['true']; m = np.isfinite(t)
        if not m.any(): continue
        sc = np.zeros(w['cand'].shape[0])
        for term, wt in weights.items():
            sc = sc + wt * zwell(PHYS[wi][term])
        al = allowed_idx(wi, restrict_R)
        sel = al[np.argsort(sc[al])[:k]]
        errs.append(w['cand'][sel].mean(0)[m] - t[m])
    return pooled_rmse(errs)

def oracle_overall(restrict_R=None):
    """対象候補内での oracle(真値最良1本)。restrict_R 内に正解が残っているかの確認。"""
    errs = []; navg = []
    for wi, w in enumerate(WELLS):
        t = w['true']; m = np.isfinite(t)
        if not m.any(): continue
        al = allowed_idx(wi, restrict_R); navg.append(len(al))
        rc = np.sqrt(np.mean((w['cand'][al][:, m] - t[None, m]) ** 2, 1))
        errs.append(w['cand'][al[int(np.argmin(rc))], m] - t[m])
    return pooled_rmse(errs), float(np.mean(navg))

In [ ]:
# ===== 比較: 単項ごと / 組合せ / top-k =====
print(f'基準:  oracle={ORACLE:.3f}  lik-ensemble={LIKENS:.3f}\n')
print('-- 物理項 単独で選択 (top1) --')
single = {tm: select_overall({tm: 1.0}, k=1) for tm in TERMS}
for tm, v in sorted(single.items(), key=lambda x: x[1]):
    print(f'  {tm:8s} = {v:.3f}')

print('\n-- 組合せ(等重み) --')
for name, wts in {
    'all-equal': {tm: 1.0 for tm in TERMS},
    'smooth+dip': {'rough': 1.0, 'wiggle': 1.0, 'dipdev': 1.0, 'dipvar': 1.0},
    'dip+anchor': {'dipdev': 1.0, 'dipvar': 1.0, 'anchor': 1.0, 'clip': 1.0},
}.items():
    for k in (1, 5):
        print(f'  {name:12s} top{k} = {select_overall(wts, k):.3f}')

# 上位の単項を貪欲に重ね掛け(簡易・重みは学習しない=過学習しにくい)
print('\n-- 貪欲に追加 (top1) --')
order = sorted(TERMS, key=lambda t: single[t]); cur = {}
best = 1e9
for tm in order:
    trial = dict(cur, **{tm: 1.0}); v = select_overall(trial, 1)
    if v < best - 1e-3:
        best = v; cur = trial; print(f'  + {tm:8s} -> {v:.3f}')
print('採用項:', list(cur), 'best top1 =', round(best, 3))

In [ ]:
# ===== lik-ensemble から RMSE<=R の候補だけに絞って物理選択 =====
R = 5.0
orc_all, n_all = oracle_overall(None)
orc_r, n_r = oracle_overall(R)
print(f'対象候補内 oracle: 全候補={orc_all:.3f}(平均{n_all:.0f}本)  |  ens±{R}内={orc_r:.3f}(平均{n_r:.1f}本)')
print('  判定: ens±R内 oracle が全候補oracleより大きく悪化 = 正解はensから遠い少数派 → 絞るのは有害。')
print('  (実測: 4.62 → 8.70 と悪化。よって ens近傍への限定は不可。記録用に下も出すが非推奨)\n')

print(f'-- 物理項 単独 (top1, ens±{R}内に限定) --')
for tm, v in sorted({tm: select_overall({tm: 1.0}, 1, R) for tm in TERMS}.items(), key=lambda x: x[1]):
    print(f'  {tm:8s} = {v:.3f}')

print(f'\n-- 組合せ (ens±{R}内に限定) --')
for name, wts in {
    'all-equal': {tm: 1.0 for tm in TERMS},
    'smooth+dip': {'rough': 1.0, 'wiggle': 1.0, 'dipdev': 1.0, 'dipvar': 1.0},
    'dip+anchor': {'dipdev': 1.0, 'dipvar': 1.0, 'anchor': 1.0, 'clip': 1.0},
}.items():
    for k in (1, 5):
        print(f'  {name:12s} top{k} = {select_overall(wts, k, R):.3f}')
print('\n結論: 正解候補は consensus(ens)から離れた少数派。physics/ens距離では選べない → 学習セレクタの効きが鍵。')

In [ ]:
# ===== 可視化: 物理選択 vs 真値 vs oracle vs 最滑(rough) =====
def plot_well(key, weights=None):
    w = WELLS[key] if isinstance(key, int) else next(x for x in WELLS if x['wid'] == key)
    wi = WELLS.index(w); cand = w['cand']; t = w['true']; m = np.isfinite(t); x = np.arange(cand.shape[1])
    rc = np.sqrt(np.mean((cand[:, m] - t[None, m]) ** 2, 1)); bo = int(np.argmin(rc))
    if weights is None: weights = {'rough': 1.0, 'wiggle': 1.0, 'dipdev': 1.0, 'dipvar': 1.0}
    sc = np.zeros(cand.shape[0])
    for tm, wt in weights.items(): sc = sc + wt * zwell(PHYS[wi][tm])
    ph = int(np.argmin(sc)); sm = int(np.argmin(PHYS[wi]['rough']))
    rmse = lambda c: np.sqrt(np.mean((c[m] - t[m]) ** 2))
    plt.figure(figsize=(13, 5))
    plt.plot(x, cand.T, color='red', alpha=0.05, lw=0.6)
    plt.plot(x, t, 'k', lw=2, label='true')
    plt.plot(x, cand[bo], 'lime', lw=2, label=f'oracle ({rc[bo]:.2f})')
    plt.plot(x, cand[ph], 'b', lw=2, label=f'physics ({rmse(cand[ph]):.2f})')
    plt.plot(x, cand[sm], 'orange', lw=1.5, label=f'smoothest ({rmse(cand[sm]):.2f})')
    plt.title(f"{w['wid']}"); plt.xlabel('eval row'); plt.legend(); plt.tight_layout(); plt.show()

for i in [0, 1, 2, 3]:
    plot_well(i)